# Does the ranking predict, or only describe?

`merchant_ranking_baseline.ipynb` ranks merchants on the whole 20-month window. That
tells us who *was* good. Onboarding is a decision about who *will be* good, and nothing
in that notebook tests whether the ranking can do that.

This notebook runs the test.

| | Period | Used for |
| --- | --- | --- |
| **Train** | 28 Feb 2021 – 28 Feb 2022 | build features, score, pick a predicted top 100 |
| **Test** | 1 Mar 2022 – 31 Aug 2022 | measure what those merchants actually did next |

The model never sees a single test-period transaction. September and October 2022 are
left out of both windows so the test period is a clean six months.

Three questions:

1. **Does the score track reality?** Spearman correlation between predicted rank and
   realised revenue.
2. **Does it beat the obvious baseline?** The baseline is "rank by past revenue". If we
   cannot beat it, the extra nine features are not earning their keep.
3. **How much money would the pick have captured?** Of the revenue the best possible
   100 merchants would have produced, what share did our 100 produce?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "tables" / "processed"

TRAIN_START, TRAIN_END = "2021-02-28", "2022-02-28"
TEST_START,  TEST_END  = "2022-03-01", "2022-08-31"

spark = (
    SparkSession.builder.appName("MAST30034 Project 2 - Time split validation")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

transactions = spark.read.parquet(str(PROCESSED / "transactions_clean"))
merchants = spark.read.parquet(str(PROCESSED / "merchants_clean"))
consumers = spark.read.parquet(str(PROCESSED / "consumers_sa2"))

train = transactions.filter(F.col("order_datetime").between(TRAIN_START, TRAIN_END))
test  = transactions.filter(F.col("order_datetime").between(TEST_START,  TEST_END))

print(f"train : {train.count():,} transactions  ({TRAIN_START} to {TRAIN_END})")
print(f"test  : {test.count():,} transactions  ({TEST_START} to {TEST_END})")
print(f"held out entirely: {transactions.count() - train.count() - test.count():,}")

## 1. Rebuild the features on the training window only

The feature table from `01_data_quality_eda` is computed over the whole 20 months, so it
cannot be used here — it already contains test-period information. Every feature below
is recomputed from training transactions alone, using the same definitions.

Growth inside the training window compares the **first three months** of a merchant's
training activity with the **last three months**, the same shape as `log_value_growth`.

In [ ]:
def build_features(tx, months_in_window):
    core = (
        tx.groupBy("merchant_abn")
        .agg(
            F.sum("dollar_value").alias("gmv"),
            F.count("*").alias("transaction_count"),
            F.countDistinct("user_id").alias("unique_customers"),
            F.countDistinct(F.trunc("order_datetime", "month")).alias("active_months"),
        )
        .withColumn("transactions_per_customer",
                    F.col("transaction_count") / F.col("unique_customers"))
        .withColumn("active_month_ratio", F.col("active_months") / F.lit(months_in_window))
    )

    per_cust = (tx.groupBy("merchant_abn", "user_id")
                  .agg(F.sum("dollar_value").alias("cust_gmv"),
                       F.count("*").alias("cust_orders")))

    conc = (per_cust.groupBy("merchant_abn")
            .agg((F.sum(F.when(F.col("cust_orders") > 1, 1).otherwise(0)) / F.count("*"))
                     .alias("repeat_rate"),
                 F.sum(F.pow("cust_gmv", 2)).alias("_sq"),
                 F.sum("cust_gmv").alias("_tot"))
            .withColumn("customer_hhi", F.col("_sq") / F.pow(F.col("_tot"), 2))
            .drop("_sq", "_tot"))

    monthly = (tx.withColumn("month", F.trunc("order_datetime", "month"))
                 .groupBy("merchant_abn", "month")
                 .agg(F.sum("dollar_value").alias("monthly_gmv")))

    stability = (monthly.groupBy("merchant_abn")
                 .agg(F.avg("monthly_gmv").alias("mean_monthly_value"),
                      F.stddev_pop("monthly_gmv").alias("sd_monthly_value"))
                 .withColumn("value_cv",
                             F.col("sd_monthly_value") / F.col("mean_monthly_value")))

    # growth: first 3 active months vs last 3 active months, within the window
    w_asc = Window.partitionBy("merchant_abn").orderBy(F.asc("month"))
    w_desc = Window.partitionBy("merchant_abn").orderBy(F.desc("month"))
    ranked_months = (monthly
        .withColumn("i_early", F.row_number().over(w_asc))
        .withColumn("i_late", F.row_number().over(w_desc)))

    growth = (ranked_months.groupBy("merchant_abn")
        .agg(F.sum(F.when(F.col("i_early") <= 3, F.col("monthly_gmv"))).alias("early_3m_value"),
             F.sum(F.when(F.col("i_late") <= 3, F.col("monthly_gmv"))).alias("recent_3m_value"),
             F.count("*").alias("n_months"))
        .withColumn("log_value_growth",
                    F.when(F.col("n_months") >= 6,
                           F.log1p("recent_3m_value") - F.log1p("early_3m_value")))
        .drop("n_months"))

    cust_profile = per_cust.join(
        consumers.select("user_id", "state", "sa2_share",
                         "abs_median_household_income_weekly", "seifa_irsad_decile"),
        on="user_id", how="left")
    has_income = F.col("abs_median_household_income_weekly").isNotNull()
    has_irsad = F.col("seifa_irsad_decile").isNotNull()
    wgt = F.coalesce(F.col("sa2_share"), F.lit(1.0))

    base = (cust_profile.groupBy("merchant_abn")
        .agg((F.sum(F.when(has_income, F.col("abs_median_household_income_weekly") * wgt))
              / F.sum(F.when(has_income, wgt))).alias("customer_income"),
             (F.sum(F.when(has_irsad, F.col("seifa_irsad_decile") * wgt))
              / F.sum(F.when(has_irsad, wgt))).alias("customer_irsad")))

    return (core.join(conc, "merchant_abn", "left")
                .join(stability.select("merchant_abn", "value_cv"), "merchant_abn", "left")
                .join(growth, "merchant_abn", "left")
                .join(base, "merchant_abn", "left")
                .join(merchants.select("merchant_abn", "name", "take_rate"),
                      "merchant_abn", "left")
                .withColumn("estimated_bnpl_revenue",
                            F.col("gmv") * F.col("take_rate") / 100.0))


train_features = build_features(train, months_in_window=12).filter(F.col("take_rate").isNotNull())
train_pd = train_features.toPandas()
print(f"{len(train_pd):,} merchants rankable on the training window")
train_pd.head(3)

## 2. Score them — exactly the weights from the baseline notebook

Same weights, same shrinkage, same "a missing feature scores 50" rule. Nothing is tuned
against the test period; that would defeat the point of the exercise.

In [ ]:
K = 300
WEIGHTS = {
    "bnpl_revenue_shrunk":       0.30,
    "growth_shrunk":             0.20,
    "transactions_per_customer": 0.10,
    "stability":                 0.10,
    "unique_customers":          0.05,
    "active_month_ratio":        0.05,
    "repeat_rate":               0.05,
    "customer_income":           0.05,
    "customer_irsad":            0.05,
    "customer_spread":           0.05,
}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-9


def score(df):
    d = df.copy()
    d["shrinkage"] = d["transaction_count"] / (d["transaction_count"] + K)
    for raw, shrunk in [("estimated_bnpl_revenue", "bnpl_revenue_shrunk"),
                        ("log_value_growth", "growth_shrunk")]:
        d[shrunk] = d["shrinkage"] * d[raw] + (1 - d["shrinkage"]) * d[raw].mean()
    d["stability"] = -d["value_cv"]
    d["customer_spread"] = 1 - d["customer_hhi"]
    ranks = pd.DataFrame(
        {c: d[c].rank(pct=True, na_option="keep") * 100 for c in WEIGHTS}
    ).fillna(50.0)
    d["score"] = sum(ranks[c] * w for c, w in WEIGHTS.items())
    return d


train_scored = score(train_pd)
train_scored["predicted_rank"] = train_scored["score"].rank(ascending=False,
                                                            method="first").astype(int)
train_scored[["predicted_rank", "name", "estimated_bnpl_revenue", "score"]] \
    .sort_values("predicted_rank").head(10).round(2)

## 3. What actually happened next

Realised revenue to the firm over the six test months: `test GMV × take_rate / 100`.
Merchants with no test-period activity are kept with a realised revenue of 0 — walking
away is exactly the outcome a ranking should have avoided, so hiding those rows would
flatter the result.

In [ ]:
realised = (
    test.groupBy("merchant_abn")
        .agg(F.sum("dollar_value").alias("test_gmv"),
             F.count("*").alias("test_transactions"))
        .join(merchants.select("merchant_abn", "take_rate"), "merchant_abn", "left")
        .withColumn("realised_revenue", F.col("test_gmv") * F.col("take_rate") / 100.0)
        .select("merchant_abn", "test_gmv", "test_transactions", "realised_revenue")
        .toPandas()
)

ev = train_scored.merge(realised, on="merchant_abn", how="left")
ev[["test_gmv", "test_transactions", "realised_revenue"]] = \
    ev[["test_gmv", "test_transactions", "realised_revenue"]].fillna(0.0)
ev["realised_rank"] = ev["realised_revenue"].rank(ascending=False, method="first").astype(int)

print(f"merchants scored on train  : {len(ev):,}")
print(f"still trading in test      : {(ev['test_transactions'] > 0).sum():,}")
print(f"stopped entirely           : {(ev['test_transactions'] == 0).sum():,}")
print(f"total realised revenue     : ${ev['realised_revenue'].sum()/1e6:,.1f}M over 6 months")

## 4. The three questions

**Spearman correlation** is the correlation of the *ranks*, not the raw values: 1.0 is a
perfect ordering, 0 is noise. Rank correlation is the right measure here because the
ranking's job is to order merchants, not to predict a dollar figure.

**Capture rate** answers the money question directly. If we had onboarded our predicted
100, what share of the realised revenue of the *best possible* 100 would we have
collected? 100% is unreachable in practice — it needs perfect foresight.

In [ ]:
from scipy.stats import spearmanr

ours = ev.nsmallest(100, "predicted_rank")
baseline = ev.nlargest(100, "estimated_bnpl_revenue")      # rank by past revenue
perfect = ev.nlargest(100, "realised_revenue")

rho_score, p_score = spearmanr(ev["score"], ev["realised_revenue"])
rho_base, p_base = spearmanr(ev["estimated_bnpl_revenue"], ev["realised_revenue"])

best_possible = perfect["realised_revenue"].sum()

rows = []
for label, sel in [("our score", ours), ("baseline (past revenue)", baseline),
                   ("perfect foresight", perfect)]:
    rows.append({
        "picker": label,
        "hits in realised top 100": len(set(sel["merchant_abn"]) & set(perfect["merchant_abn"])),
        "realised revenue ($M)": sel["realised_revenue"].sum() / 1e6,
        "capture rate": sel["realised_revenue"].sum() / best_possible,
        "picks that stopped trading": int((sel["test_transactions"] == 0).sum()),
    })

print(f"Spearman, our score vs realised revenue : {rho_score:.3f}  (p={p_score:.1e})")
print(f"Spearman, baseline   vs realised revenue: {rho_base:.3f}  (p={p_base:.1e})")
print()
pd.DataFrame(rows).set_index("picker").round(3)

In [ ]:
overlap_ours_base = len(set(ours["merchant_abn"]) & set(baseline["merchant_abn"]))
print(f"our top 100 vs baseline top 100 overlap: {overlap_ours_base}/100")

added = ours[~ours["merchant_abn"].isin(baseline["merchant_abn"])]
dropped = baseline[~baseline["merchant_abn"].isin(ours["merchant_abn"])]
print(f"\nmedian realised revenue of the {len(added)} merchants we added   : "
      f"${added['realised_revenue'].median():,.0f}")
print(f"median realised revenue of the {len(dropped)} merchants we dropped : "
      f"${dropped['realised_revenue'].median():,.0f}")
print("\n(if the added merchants earn more than the dropped ones, the extra features paid off)")

## 5b. Which features actually predicted anything?

The score is a blend, so when it underperforms the question is *which ingredient*. Each
training feature is correlated on its own against realised revenue, then the score is
rebuilt under several weightings to see which of them would have picked better.

In [ ]:
feature_cols = ["estimated_bnpl_revenue", "gmv", "transaction_count", "unique_customers",
                "transactions_per_customer", "repeat_rate", "value_cv",
                "active_month_ratio", "log_value_growth", "customer_income",
                "customer_irsad", "customer_hhi", "take_rate"]

rows = []
for c in feature_cols:
    s = ev[c].fillna(ev[c].median())
    rows.append({"feature": c, "spearman vs realised revenue": spearmanr(s, ev["realised_revenue"])[0]})

pd.DataFrame(rows).set_index("feature").sort_values(
    "spearman vs realised revenue", key=abs, ascending=False).round(3)

In [ ]:
ranks_ev = pd.DataFrame(
    {c: ev[c].rank(pct=True, na_option="keep") * 100 for c in WEIGHTS}
).fillna(50.0)
best_possible = perfect["realised_revenue"].sum()


def try_weights(label, weights):
    total = sum(weights.values())
    s = sum(ranks_ev[c] * (w / total) for c, w in weights.items())
    sel = ev.assign(_s=s).nlargest(100, "_s")
    return {
        "weighting": label,
        "spearman": spearmanr(s, ev["realised_revenue"])[0],
        "capture rate": sel["realised_revenue"].sum() / best_possible,
        "hits in realised top 100": len(set(sel["merchant_abn"]) & set(perfect["merchant_abn"])),
    }


variants = [
    try_weights("current score (10 features)", WEIGHTS),
    try_weights("revenue only (shrunk)", {"bnpl_revenue_shrunk": 1.0}),
    try_weights("revenue 70 / growth 30",
                {"bnpl_revenue_shrunk": 0.7, "growth_shrunk": 0.3}),
    try_weights("revenue 60 / growth 20 / engagement 20",
                {"bnpl_revenue_shrunk": 0.6, "growth_shrunk": 0.2,
                 "transactions_per_customer": 0.1, "repeat_rate": 0.1}),
    try_weights("current score, no ABS features",
                {k: v for k, v in WEIGHTS.items()
                 if k not in ("customer_income", "customer_irsad")}),
]

# unshrunk past revenue - the plain baseline, expressed the same way
plain = ev["estimated_bnpl_revenue"].rank(pct=True) * 100
sel = ev.assign(_s=plain).nlargest(100, "_s")
variants.append({
    "weighting": "past revenue, no shrinkage",
    "spearman": spearmanr(plain, ev["realised_revenue"])[0],
    "capture rate": sel["realised_revenue"].sum() / best_possible,
    "hits in realised top 100": len(set(sel["merchant_abn"]) & set(perfect["merchant_abn"])),
})

pd.DataFrame(variants).set_index("weighting").round(3)

## 5. Pictures

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GREY = "#4C78A8", "#F58518", "#8C8C8C"

fig, ax = plt.subplots(figsize=(6.4, 6))
ax.scatter(ev["predicted_rank"], ev["realised_rank"], s=6, alpha=0.25, color=GREY,
           edgecolors="none")
ax.scatter(ours["predicted_rank"], ours["realised_rank"], s=14, color=ORANGE,
           edgecolors="none", label="our predicted top 100")
lim = len(ev)
ax.plot([1, lim], [1, lim], color=BLUE, linewidth=1.5, label="perfect prediction")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.invert_xaxis(); ax.invert_yaxis()      # rank 1 at top-right of each axis
ax.set_xlabel("predicted rank (train window)")
ax.set_ylabel("realised rank (test window)")
ax.set_title(f"Predicted vs realised rank - Spearman {rho_score:.2f}")
ax.legend(loc="lower left", frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
caps = pd.DataFrame(rows).set_index("picker")["capture rate"] * 100

fig, ax = plt.subplots(figsize=(7, 2.8))
colors = [ORANGE, GREY, BLUE]
bars = ax.barh(caps.index[::-1], caps.values[::-1], color=colors[::-1], height=0.6)
ax.bar_label(bars, labels=[f"{v:.0f}%" for v in caps.values[::-1]], padding=4, fontsize=9)
ax.set_xlim(0, 105)
ax.set_xlabel("% of the realised revenue a perfect top 100 would have earned")
ax.set_title("How much of the achievable revenue each picker captured")
plt.tight_layout(); plt.show()

## 6. Reading the result

Fill this in once the numbers are on screen — the honest version, whichever way it goes:

- **Spearman above ~0.5** means the ranking genuinely carries information about next
  year. Near 0 would mean it describes the past and nothing more.
- **Capture rate above the baseline** is the claim that matters commercially: the extra
  features earned real money relative to just ranking by past revenue. If it is *below*
  the baseline, say so and simplify the model — that is a finding, not a failure.
- **Picks that stopped trading** is the most expensive kind of mistake: a merchant
  onboarded, then gone. Compare ours against the baseline's.

Two limitations to state whichever way the numbers land:

1. Six months is one test window, not a repeated experiment. A different split could
   give a different number.
2. The test period sits inside the third transaction snapshot. If that file has its own
   character — a different mix of merchants, say — some of what looks like predictive
   power (or the lack of it) belongs to the data rather than the model.